# Capítulo 9: k-Vizinhos Mais Próximos

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 12 de Grus (2019).

> Se você quiser irritar os seus vizinhos, conte a verdade sobre eles.
>
> — Pietro Aretino

Quase todo modelo deste livro olha o conjunto de dados inteiro para aprender um padrão: ajusta coeficientes, mede erros, itera. O k-vizinhos mais próximos não faz nada disso. Ele não aprende — ele guarda. Na hora de classificar um ponto novo, procura os pontos rotulados mais parecidos e deixa que eles votem.

É o modelo mais simples deste livro. Os capítulos 5 e 8 já fizeram você implementar peças do zero — gradiente descendente, funções de avaliação —, mas este é o primeiro *classificador* completo que você monta do dado bruto até a previsão. Ele precisa de exatamente duas coisas: uma noção de distância, e a hipótese de que pontos próximos se parecem.

Ao final deste capítulo, você será capaz de:

- Explicar o que o k-vizinhos faz e o que ele deliberadamente ignora
- Implementar uma votação majoritária que resolve empates de forma determinística
- Classificar dados reais com o algoritmo que você escreveu
- Explicar por que aumentar o número de dimensões degrada o método, e demonstrar isso numericamente
- Reconhecer o mesmo algoritmo na interface do `scikit-learn`

## Seções

| Seção | Tópico |
|---|---|
| [9.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-o-modelo.html) | O Modelo |
| [9.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html) | Exemplo: O Dataset Iris |
| [9.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/03-a-maldicao-da-dimensionalidade.html) | A Maldição da Dimensionalidade |

## O Modelo

> **📌 Nota**
>
> Esta seção corresponde a *The Model*, do capítulo 12 de Grus (2019).

Imagine que você quer prever em quem uma pessoa vai votar. Sem saber mais nada sobre ela, uma aposta razoável é olhar como os vizinhos dela votam. Se você souber mais — idade, renda, quantos filhos tem —, dá para olhar os vizinhos *naquelas dimensões* também, e não só na geográfica.

Essa é a ideia inteira.

> **🔷 Conceito**
>
> O k-vizinhos mais próximos precisa de apenas duas coisas:
>
> 1. Uma noção de **distância**
> 2. A hipótese de que pontos **próximos são parecidos**
>
> Ele não faz nenhuma suposição matemática sobre a forma dos dados e não tem etapa de treino. Em compensação, **ignora quase toda a informação disponível**: a previsão para um ponto novo depende só do punhado de pontos mais próximos dele.

Essa é uma troca real, não um detalhe. O k-vizinhos raramente ajuda a *entender* o fenômeno. Prever o voto de alguém a partir do voto dos vizinhos não diz nada sobre a causa daquele voto — enquanto um modelo baseado em renda e estado civil poderia dizer.

### Contando votos

Escolhido um *k* — 3 ou 5, digamos —, classificar um ponto novo é achar os *k* pontos rotulados mais próximos e deixá-los votar. Precisamos, então, de uma função que conte votos.

A primeira tentativa é direta:

In [ ]:
from typing import List
from collections import Counter

def raw_majority_vote(labels: List[str]) -> str:
    votes = Counter(labels)
    winner, _ = votes.most_common(1)[0]
    return winner

assert raw_majority_vote(['a', 'b', 'c', 'b']) == 'b'

Só que ela não faz nada de inteligente com empates. Se estivéssemos classificando filmes e os cinco mais próximos fossem G, G, PG, PG e R, teríamos dois votos para G e dois para PG. Há três saídas possíveis:

- Escolher um dos vencedores ao acaso
- Ponderar os votos pela distância e pegar o vencedor ponderado
- Reduzir *k* até haver um vencedor único

Vamos implementar a terceira:

In [ ]:
def majority_vote(labels: List[str]) -> str:
    """Assume que os rótulos estão ordenados do mais próximo ao mais distante."""
    vote_counts = Counter(labels)
    winner, winner_count = vote_counts.most_common(1)[0]
    num_winners = len([count
                       for count in vote_counts.values()
                       if count == winner_count])

    if num_winners == 1:
        return winner                     # vencedor único
    else:
        return majority_vote(labels[:-1])  # tenta de novo sem o mais distante

# Empate: olha os 4 primeiros, então 'b'
assert majority_vote(['a', 'b', 'c', 'b', 'a']) == 'b'

> **🟩 Exemplo**
>
> A recursão sempre termina. No pior caso, descartamos um rótulo por vez até sobrar um só — e aí ele vence sozinho.
>
> Repare que o argumento precisa estar **ordenado do mais próximo ao mais distante**. Descartar "o último" só faz sentido se o último for o vizinho menos relevante. Uma lista fora de ordem produz uma resposta errada sem erro nenhum.

### O classificador

Com a votação pronta, o classificador cabe em poucas linhas. A função `distance` vem do capítulo 4:

In [ ]:
from typing import NamedTuple
from scratch.linear_algebra import Vector, distance

class LabeledPoint(NamedTuple):
    point: Vector
    label: str

def knn_classify(k: int,
                 labeled_points: List[LabeledPoint],
                 new_point: Vector) -> str:

    # Ordena os pontos rotulados do mais próximo ao mais distante.
    by_distance = sorted(labeled_points,
                         key=lambda lp: distance(lp.point, new_point))

    # Pega os rótulos dos k mais próximos...
    k_nearest_labels = [lp.label for lp in by_distance[:k]]

    # ...e deixa que votem.
    return majority_vote(k_nearest_labels)

É isso: ordenar por distância, cortar em *k*, contar. Não há treino, não há parâmetros ajustados, não há otimização. O modelo *é* o conjunto de dados.

> **💡 Dica — Na prática: `scikit-learn`**
>
> Você acabou de escrever o algoritmo. Na vida real, você usaria isto:
>
> ```python
> from sklearn.neighbors import KNeighborsClassifier
>
> modelo = KNeighborsClassifier(n_neighbors=5)
> modelo.fit(X_treino, y_treino)
> modelo.predict(X_novo)
> ```
>
> Sobre o custo: o `knn_classify` acima ordena a lista inteira de pontos rotulados a cada consulta, e `sorted()` sobre $n$ pontos custa $O(n \log n)$ — isso além do $O(n \cdot d)$ de calcular as $n$ distâncias, cada uma envolvendo $d$ dimensões. Não precisávamos ordenar tudo: bastaria manter um heap com os $k$ mais próximos para cair para $O(n \log k)$. Não fizemos isso porque o objetivo aqui era clareza, não desempenho — mas é a otimização óbvia se este código fosse para produção.
>
> Sobre o desempate: o `scikit-learn` não reduz *k* como fizemos. Ele toma a moda dos rótulos codificados, e como o atributo `classes_` fica ordenado, o desempate favorece o rótulo alfabeticamente menor — não é a mesma regra da nossa recursão, mas é igualmente arbitrária. Ele também aceita pesos por distância (`weights='distance'`).
>
> Sobre a busca: por padrão (`algorithm='auto'`), o `scikit-learn` escolhe entre força bruta e estruturas de indexação como `KDTree` ou `BallTree`, que evitam comparar o ponto novo com todos os outros. Mas essas estruturas só ajudam em dimensão baixa — em dimensão alta elas degradam para força bruta, o mesmo custo que o nosso código sempre teve. O motivo é o assunto da seção 9.3: a partir de um certo número de dimensões, nem a distância nem a estrutura que a indexa continuam ajudando.
>
> Nada disso muda o que o modelo *é*. É a mesma ordenação por distância seguida de votação que você implementou acima.

## Exemplo: O Dataset Iris

> **📌 Nota**
>
> Esta seção corresponde a *Example: The Iris Dataset*, do capítulo 12 de Grus (2019).

O *Iris* é um clássico do aprendizado de máquina. São 150 flores de três espécies, e para cada uma temos quatro medidas: comprimento e largura da pétala, comprimento e largura da sépala. A tarefa é prever a espécie a partir das quatro medidas.

A fonte original é o repositório da UCI:

```python
import requests

data = requests.get(
  "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
)

with open('iris.dat', 'w') as f:
    f.write(data.text)
```

> **❗ Importante**
>
> Esse download **não é executado** aqui: o arquivo já está salvo em `dados/iris.data`, e é dele que vamos ler.
>
> Uma análise que baixa o dado de novo a cada execução é frágil. Basta o servidor sair do ar, mudar de endereço ou passar a servir uma versão diferente do arquivo para o resultado deixar de ser reproduzível — e você não tem como saber qual das três coisas aconteceu. Baixe uma vez, guarde a cópia, trabalhe em cima dela. O código do download continua valendo a pena registrar, porque saber de onde o dado veio *é* parte da análise.

Os dados são separados por vírgula, com os campos:

```
sepal_length, sepal_width, petal_length, petal_width, class
```

A primeira linha, por exemplo:

```
5.1,3.5,1.4,0.2,Iris-setosa
```

### Carregando os dados

Nossa função de vizinhos espera `LabeledPoint`, então é assim que vamos representar cada flor:

In [ ]:
from typing import Dict, List
import csv
from collections import defaultdict
from scratch.linear_algebra import Vector
from scratch.k_nearest_neighbors import LabeledPoint, knn_classify

def parse_iris_row(row: List[str]) -> LabeledPoint:
    """
    sepal_length, sepal_width, petal_length, petal_width, class
    """
    measurements = [float(value) for value in row[:-1]]
    # a classe vem como "Iris-virginica"; queremos só "virginica"
    label = row[-1].split("-")[-1]

    return LabeledPoint(measurements, label)

with open('dados/iris.data') as f:
    reader = csv.reader(f)
    iris_data = [parse_iris_row(row) for row in reader if row]

# Agrupamos também por espécie, para poder desenhar
points_by_species: Dict[str, List[Vector]] = defaultdict(list)
for iris in iris_data:
    points_by_species[iris.label].append(iris.point)

len(iris_data), sorted(points_by_species)

> **📌 Nota**
>
> Dois detalhes do carregamento:
>
> - O `if row` no final da compreensão descarta a linha em branco no fim do arquivo, que faria o `parse_iris_row` estourar. Arquivo de dado real quase sempre tem uma dessas.
> - `LabeledPoint` e `knn_classify` não são redefinidos aqui: vêm de `scratch.k_nearest_neighbors`, o mesmo código que você escreveu na [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-o-modelo.html).

### Olhando os dados

Gostaríamos de visualizar as medidas para ver como variam por espécie. O problema é que são quatro dimensões, o que dificulta o desenho. Uma saída é olhar os gráficos de dispersão para cada um dos seis pares de medidas:

In [ ]:
# Figura: Dispersões do Iris, para os seis pares de medidas
from matplotlib import pyplot as plt

metrics = ['comprimento sépala', 'largura sépala',
           'comprimento pétala', 'largura pétala']
pairs = [(i, j) for i in range(4) for j in range(4) if i < j]
marks = ['+', '.', 'x']  # três classes, três marcadores

fig, ax = plt.subplots(2, 3, figsize=(10, 6))

for row in range(2):
    for col in range(3):
        i, j = pairs[3 * row + col]
        ax[row][col].set_title(f"{metrics[i]} vs {metrics[j]}", fontsize=8)
        ax[row][col].set_xticks([])
        ax[row][col].set_yticks([])

        for mark, (species, points) in zip(marks, points_by_species.items()):
            xs = [point[i] for point in points]
            ys = [point[j] for point in points]
            ax[row][col].scatter(xs, ys, marker=mark, label=species)

ax[-1][-1].legend(loc='lower right', prop={'size': 6})
plt.tight_layout()
plt.show()

As medidas realmente se agrupam por espécie. Olhando só para as sépalas, seria difícil separar *versicolor* de *virginica* — mas quando entram comprimento e largura da pétala, a separação fica clara. É exatamente a situação em que vizinhos mais próximos funciona bem.

### Classificando

Primeiro dividimos os dados em treino e teste:

In [ ]:
import random
from scratch.machine_learning import split_data

random.seed(12)
iris_train, iris_test = split_data(iris_data, 0.70)

assert len(iris_train) == 0.7 * 150
assert len(iris_test) == 0.3 * 150

len(iris_train), len(iris_test)

> **⚠️ Atenção — A semente não é opcional**
>
> `random.seed(12)` fixa o embaralhamento que o `split_data` faz. Sem ela, cada execução separa um conjunto de treino diferente — e com ele vêm uma acurácia diferente e uma matriz de confusão diferente. Você não conseguiria repetir o próprio resultado, nem comparar duas escolhas de *k* sabendo que a diferença veio do *k* e não do sorteio.
>
> Fixe a semente em todo experimento que usa aleatoriedade. Aqui é o `random` da biblioteca padrão, que é o módulo que o `split_data` embaralha por dentro.

Os pontos de treino são os "vizinhos" que usaremos para classificar os pontos de teste. Falta escolher *k*. Pequeno demais (pense em *k* = 1) e os outliers têm influência exagerada; grande demais (pense em *k* = 105) e simplesmente prevemos a classe mais comum do conjunto. Numa aplicação real criaríamos um conjunto de validação para escolher; aqui vamos usar *k* = 5:

In [ ]:
from typing import Tuple

# quantas vezes vimos cada par (previsto, real)
confusion_matrix: Dict[Tuple[str, str], int] = defaultdict(int)
num_correct = 0

for iris in iris_test:
    predicted = knn_classify(5, iris_train, iris.point)
    actual = iris.label

    if predicted == actual:
        num_correct += 1

    confusion_matrix[(predicted, actual)] += 1

pct_correct = num_correct / len(iris_test)
pct_correct

In [ ]:
for (previsto, real), n in sorted(confusion_matrix.items()):
    marca = "" if previsto == real else "   <-- erro"
    print(f"previsto {previsto:12s} real {real:12s} {n:3d}{marca}")

Neste conjunto simples, o modelo acerta quase tudo. Há uma *versicolor* classificada como *virginica* — justamente o par que os gráficos de dispersão mostraram ser o mais difícil de separar — e o resto sai certo.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O mesmo experimento, com a biblioteca:
>
> ```python
> from sklearn.model_selection import train_test_split
> from sklearn.neighbors import KNeighborsClassifier
> from sklearn.metrics import confusion_matrix, accuracy_score
>
> X = [p.point for p in iris_data]
> y = [p.label for p in iris_data]
>
> X_treino, X_teste, y_treino, y_teste = train_test_split(
>     X, y, test_size=0.3, random_state=12
> )
>
> modelo = KNeighborsClassifier(n_neighbors=5).fit(X_treino, y_treino)
> previsto = modelo.predict(X_teste)
>
> accuracy_score(y_teste, previsto)
> confusion_matrix(y_teste, previsto)
> ```
>
> Sete linhas contra as trinta que escrevemos — e a acurácia não sai idêntica, porque o `train_test_split` embaralha de outro jeito e o desempate é diferente. A seção anterior mostrou onde o `scikit-learn` realmente ganha: indexação, pesos por distância, escala. Neste dataset — 150 pontos, 4 dimensões — nada disso pesa; força bruta em Python puro já é instantânea. Aqui o que a biblioteca economiza é só digitação. A vantagem de verdade aparece quando os dados crescem, não neste exemplo.

## A Maldição da Dimensionalidade

> **📌 Nota**
>
> Esta seção corresponde a *The Curse of Dimensionality*, do capítulo 12 de Grus (2019).

O k-vizinhos tem um problema sério em dimensões altas, e o nome dele é **maldição da dimensionalidade**. A raiz é simples de enunciar: espaços de dimensão alta são *vastos*. Pontos neles tendem a não estar perto de ninguém.

Dá para ver isso experimentalmente. Vamos gerar pares de pontos aleatórios num "cubo unitário" de dimensão *d*, para vários valores de *d*, e medir as distâncias.

Gerar pontos aleatórios já deve ser natural a esta altura:

In [ ]:
import random
from typing import List
from scratch.linear_algebra import Vector, distance

def random_point(dim: int) -> Vector:
    return [random.random() for _ in range(dim)]

def random_distances(dim: int, num_pairs: int) -> List[float]:
    return [distance(random_point(dim), random_point(dim))
            for _ in range(num_pairs)]

Para cada dimensão de 1 a 100, calculamos 10.000 distâncias e guardamos a média e a mínima:

In [ ]:
# Figura: Distância média e mínima entre pontos aleatórios, por dimensão
import tqdm
from matplotlib import pyplot as plt

dimensions = range(1, 101)

avg_distances = []
min_distances = []

random.seed(0)
for dim in tqdm.tqdm(dimensions, desc="Maldição da dimensionalidade"):
    distances = random_distances(dim, 10000)
    avg_distances.append(sum(distances) / 10000)
    min_distances.append(min(distances))

plt.plot(dimensions, avg_distances, label='distância média')
plt.plot(dimensions, min_distances, label='distância mínima')
plt.xlabel("# de dimensões")
plt.ylabel("distância")
plt.title("10.000 distâncias aleatórias")
plt.legend()
plt.show()

São um milhão de distâncias euclidianas calculadas em Python puro, e mesmo assim o laço termina em poucos segundos — o `tqdm.tqdm` está ali só para acompanhar o progresso enquanto ele roda. O `random.seed(0)` garante que a figura é a mesma toda vez que o experimento é repetido.

A distância média entre dois pontos cresce com a dimensão, o que já era de esperar. O que incomoda é outra coisa: a **razão** entre a menor distância e a distância média.

In [ ]:
# Figura: Razão entre a menor distância e a distância média
min_avg_ratio = [min_dist / avg_dist
                 for min_dist, avg_dist in zip(min_distances, avg_distances)]

plt.plot(dimensions, min_avg_ratio)
plt.xlabel("# de dimensões")
plt.ylabel("razão")
plt.title("Distância mínima / distância média")
plt.show()

> **🔷 Conceito**
>
> Em dimensão baixa, o ponto mais próximo está **muito** mais perto que a média — a razão fica perto de zero, e "vizinho mais próximo" significa alguma coisa.
>
> Conforme a dimensão cresce, a razão sobe assintoticamente em direção a 1: o ponto mais próximo fica cada vez mais parecido, em distância, com um ponto qualquer. No gráfico acima, com *d* indo só até 100, ela já passa de 0,75 e continua subindo — ainda não chegou perto de 1, mas a tendência é clara, e ela seguiria subindo se a dimensão continuasse crescendo. É esse limite, não o valor em *d* = 100, que importa: quando a razão está perto de 1, "vizinho mais próximo" deixa de significar coisa alguma.

A intuição por trás disso: dois pontos só estão próximos se estiverem próximos em **todas** as dimensões. Cada dimensão extra — mesmo que seja puro ruído — é mais uma oportunidade para dois pontos ficarem distantes um do outro. Com dimensão suficiente, todo mundo fica longe de todo mundo.

A ressalva importa: isso vale a menos que haja muita estrutura nos dados que os faça se comportar como se tivessem dimensão bem menor. Um conjunto de 100 colunas em que 97 são combinações lineares das outras 3 não é, de fato, um conjunto de 100 dimensões.

> **💡 Dica — Na prática: o que se faz com isso**
>
> Quando os dados têm dimensão alta demais para vizinhos mais próximos, as saídas usuais são reduzir a dimensão antes de classificar — PCA, que aparece no capítulo 7, é a mais comum — ou trocar por um modelo que não dependa de distância, como as árvores de decisão do capítulo 14.
>
> O `scikit-learn` não protege você disso. `KNeighborsClassifier` aceita 500 colunas sem reclamar, roda, devolve previsões, e elas serão ruins por um motivo que nenhuma mensagem de erro vai explicar. Saber *por quê* é o que você leva desta seção.

## Leituras adicionais

O `scikit-learn` traz muitos [modelos de vizinhos mais próximos](https://scikit-learn.org/stable/modules/neighbors.html), incluindo variantes com pesos por distância e estruturas de indexação (`KDTree`, `BallTree`) que evitam comparar o ponto novo com todos os outros.

Para o tratamento estatístico do compromisso entre viés e variância na escolha de *k*, veja Hastie et al. (2009).

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed.. Springer. 2009.